In [1]:
from IPython.display import display, HTML

display(HTML("<style>.container { width:100% !important; }</style>"))

# Lab | Natural Language Processing
### SMS: SPAM or HAM

### Let's prepare the environment

In [2]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer

- Read Data for the Fraudulent Email Kaggle Challenge
- Reduce the training set to speed up development. 

In [3]:
## Read Data for the Fraudulent Email Kaggle Challenge
data = pd.read_csv("../data/kg_train.csv",encoding='latin-1')

# Reduce the training set to speed up development. 
# Modify for final system
data = data.head(1000)
print(data.shape)
data.fillna("",inplace=True)

(1000, 2)


,text,label
0,"DEAR SIR, STRICTLY A PRIVATE BUSINESS PROPOSAL...",1
1,Will do.,0
2,Nora--Cheryl has emailed dozens of memos about...,0
3,Dear Sir=2FMadam=2C I know that this proposal ...,1
4,fyi,0
...,...,...
995,So what's the latest? It sounds contradictory ...,0
996,"TRANSFER OF 36,759,000.00 MILLION POUNDS TO YO...",1
997,Barb I will call to explain. Are you back in t...,0
998,Yang on travelNot free tonite.May work tomorrow,0


### Let's divide the training and test set into two partitions

In [12]:
from sklearn.model_selection import train_test_split

# Assuming the target column is named 'label'
# (Adjust if your column name differs — e.g., 'fraud', 'is_spam', etc.)
X = data['text']          # Feature column (email content)
y = data['label']         # Target column

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y   # Critical for imbalanced datasets
)

print("Training set size:", X_train.shape)
print("Test set size:", X_test.shape)

print("\nClass distribution in full dataset:")
print(y.value_counts(normalize=True))

print("\nClass distribution in training set:")
print(y_train.value_counts(normalize=True))

print("\nClass distribution in test set:")
print(y_test.value_counts(normalize=True))

Training set size: (800,)
Test set size: (200,)

Class distribution in full dataset:
label
0    0.558
1    0.442
Name: proportion, dtype: float64

Class distribution in training set:
label
0    0.5575
1    0.4425
Name: proportion, dtype: float64

Class distribution in test set:
label
0    0.56
1    0.44
Name: proportion, dtype: float64


## Data Preprocessing

In [13]:
import string
from nltk.corpus import stopwords
print(string.punctuation)
print(stopwords.words("english")[100:110])
from nltk.stem.snowball import SnowballStemmer
snowball = SnowballStemmer('english')

!"#$%&'()*+,-./:;<=>?@[\]^_`{|}~
['needn', "needn't", 'no', 'nor', 'not', 'now', 'o', 'of', 'off', 'on']


## Now, we have to clean the html code removing words

- First we remove inline JavaScript/CSS
- Then we remove html comments. This has to be done before removing regular tags since comments can contain '>' characters
- Next we can remove the remaining tags

In [6]:
import re
import html

# Precompile regexes (faster if applied to many rows)
RE_SCRIPT_STYLE = re.compile(r"(?is)<(script|style)\b.*?>.*?</\1>")
RE_HTML_COMMENTS = re.compile(r"(?s)<!--.*?-->")
RE_TAGS = re.compile(r"(?s)<[^>]+>")   # remaining tags
RE_WHITESPACE = re.compile(r"\s+")

def strip_html(raw: str) -> str:
    if raw is None:
        return ""
    s = str(raw)

    # 1) Remove inline JS/CSS blocks
    s = RE_SCRIPT_STYLE.sub(" ", s)

    # 2) Remove HTML comments (do before tag stripping)
    s = RE_HTML_COMMENTS.sub(" ", s)

    # 3) Remove remaining tags
    s = RE_TAGS.sub(" ", s)

    # Decode HTML entities (&nbsp;, &amp;, etc.)
    s = html.unescape(s)

    # Normalize whitespace
    s = RE_WHITESPACE.sub(" ", s).strip()
    return s

In [14]:
data["text"] = data["text"].apply(strip_html)

- Remove all the special characters
    
- Remove numbers
    
- Remove all single characters
 
- Remove single characters from the start

- Substitute multiple spaces with single space

- Remove prefixed 'b'

- Convert to Lowercase

In [18]:
import re
import contractions  # pip install contractions

RE_MULTISPACE = re.compile(r"\s+")
RE_PREFIX_B  = re.compile(r"^b\s+", re.IGNORECASE)

def clean_text(text):
    if text is None:
        return ""
    
    text = str(text)
    text = RE_PREFIX_B.sub("", text)
    
    # Lowercase first
    text = text.lower()
    
    # ✅ Expand contractions HERE
    text = contractions.fix(text)
    
    # Now safe to clean aggressively
    text = re.sub(r"\d+", " ", text)
    text = re.sub(r"[^a-z\s]", " ", text)
    text = re.sub(r"\b[a-z]\b", " ", text)
    text = re.sub(r"^[a-z]\s+", "", text)
    text = RE_MULTISPACE.sub(" ", text)
    
    return text.strip()

In [19]:
data["text"] = data["text"].apply(clean_text)

In [20]:
data.head()

,text,label
0,dear sir strictly private business proposal am...,1
1,will do,0
2,nora cheryl has emailed dozens of memos about ...,0
3,dear sir fmadam know that this proposal might ...,1
4,fyi,0


## Now let's work on removing stopwords
Remove the stopwords.

In [ ]:
from nltk.corpus import stopwords

stop_words = set(stopwords.words("english"))

In [22]:
data["text"] = data["text"].apply(
    lambda x: " ".join(
        word for word in x.split() if word not in stop_words
    )
)

In [23]:
data.head()

,text,label
0,dear sir strictly private business proposal mi...,1
1,,0
2,nora cheryl emailed dozens memos haiti weekend...,0
3,dear sir fmadam know proposal might surprise e...,1
4,fyi,0


## Tame Your Text with Lemmatization
Break sentences into words, then use lemmatization to reduce them to their base form (e.g., "running" becomes "run"). See how this creates cleaner data for analysis!

In [24]:
from nltk.stem import WordNetLemmatizer

lemmatizer = WordNetLemmatizer()

In [25]:
data["text"] = data["text"].apply(
    lambda x: " ".join(
        lemmatizer.lemmatize(word) for word in x.split()
    )
)

In [26]:
data.head()

,text,label
0,dear sir strictly private business proposal mi...,1
1,,0
2,nora cheryl emailed dozen memo haiti weekend p...,0
3,dear sir fmadam know proposal might surprise e...,1
4,fyi,0


## Bag Of Words
Let's get the 10 top words in ham and spam messages (**EXPLORATORY DATA ANALYSIS**)

In [32]:
# creating subsets for ham and spam emails
ham_text = data[data["label"] == 0]["text"]
spam_text = data[data["label"] == 1]["text"]

In [34]:
# removing empty strings from ham and spam subsets
ham_text = ham_text[ham_text.str.strip() != ""]
spam_text = spam_text[spam_text.str.strip() != ""]

In [35]:
data["label"].value_counts()

label
0    558
1    442
Name: count, dtype: int64

In [ ]:
# top 10 most common words in ham emails
from sklearn.feature_extraction.text import CountVectorizer
import pandas as pd

vectorizer_ham = CountVectorizer()
X_ham = vectorizer_ham.fit_transform(ham_text)

ham_counts = X_ham.sum(axis=0).A1
ham_words = vectorizer_ham.get_feature_names_out()

ham_top10 = (
    pd.DataFrame({"word": ham_words, "count": ham_counts})
      .sort_values("count", ascending=False)
      .head(10)
)

ham_top10

,word,count
5411,state,136
4298,pm,127
6457,would,107
4428,president,99
5849,time,95
882,call,94
3685,mr,91
3899,obama,84
4215,percent,81
5053,secretary,79


In [ ]:
# top 10 most common words in spam emails
vectorizer_spam = CountVectorizer()
X_spam = vectorizer_spam.fit_transform(spam_text)

spam_counts = X_spam.sum(axis=0).A1
spam_words = vectorizer_spam.get_feature_names_out()

spam_top10 = (
    pd.DataFrame({"word": spam_words, "count": spam_counts})
      .sort_values("count", ascending=False)
      .head(10)
)

spam_top10

,word,count
13785,money,987
228,account,899
2378,bank,801
7875,fund,782
21004,transaction,555
3139,business,514
4441,country,513
13903,mr,490
13596,million,463
21017,transfer,426


## Extra features

In [39]:
# Extra features (money marker, suspicious words, text length) for YOUR dataframe/column names

money_simbol_list = "|".join(["euro", "dollar", "pound", "€", r"\$"])
suspicious_words = "|".join([
    "free", "cheap", "sex", "money", "account", "bank", "fund",
    "transfer", "transaction", "win", "deposit", "password"
])

# 1) Money indicator (0/1)
data["money_mark"] = data["text"].str.contains(
    money_simbol_list, case=False, regex=True, na=False
).astype(int)

# 2) Suspicious words indicator (0/1)
data["suspicious_words"] = data["text"].str.contains(
    suspicious_words, case=False, regex=True, na=False
).astype(int)

# 3) Text length (number of characters)
data["text_len"] = data["text"].fillna("").astype(str).str.len()

data.head()

,text,label,money_mark,suspicious_words,text_len
0,dear sir strictly private business proposal mi...,1,1,1,1493
1,,0,0,0,0
2,nora cheryl emailed dozen memo haiti weekend p...,0,0,0,114
3,dear sir fmadam know proposal might surprise e...,1,1,1,1346
4,fyi,0,0,0,3


## How would work the Bag of Words with Count Vectorizer concept?

In [43]:
vectorizer = CountVectorizer()
X_all = vectorizer.fit_transform(data["text"])

word_counts = X_all.sum(axis=0).A1
words = vectorizer.get_feature_names_out()

top_words = (
    pd.DataFrame({"word": words, "count": word_counts})
      .sort_values("count", ascending=False)
      .head(10)
)

In [45]:
vectorizer = CountVectorizer()
X_train_counts = vectorizer.fit_transform(X_train)
X_test_counts = vectorizer.transform(X_test)

## TF-IDF

- Load the vectorizer

- Vectorize all dataset

- print the shape of the vetorized dataset

In [44]:
from sklearn.feature_extraction.text import TfidfVectorizer

# 1️⃣ Load the vectorizer
tfidf_vectorizer = TfidfVectorizer()

# 2️⃣ Fit on training data and transform
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)

# 3️⃣ Transform test data (DO NOT fit again)
X_test_tfidf = tfidf_vectorizer.transform(X_test)

# 4️⃣ Print shapes
print("Train TF-IDF shape:", X_train_tfidf.shape)
print("Test TF-IDF shape:", X_test_tfidf.shape)

Train TF-IDF shape: (800, 23568)
Test TF-IDF shape: (200, 23568)


## And the Train a Classifier?

In [ ]:
#Multinomial Naive Bayes
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# 1️⃣ Initialize model
nb_model = MultinomialNB()

# 2️⃣ Train
nb_model.fit(X_train_tfidf, y_train)

# 3️⃣ Predict
y_pred_nb = nb_model.predict(X_test_tfidf)

# 4️⃣ Evaluate
print("Naive Bayes Accuracy:", accuracy_score(y_test, y_pred_nb))
print("\nClassification Report:\n", classification_report(y_test, y_pred_nb))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred_nb))

Naive Bayes Accuracy: 0.9

Classification Report:
               precision    recall  f1-score   support

           0       1.00      0.82      0.90       112
           1       0.81      1.00      0.90        88

    accuracy                           0.90       200
   macro avg       0.91      0.91      0.90       200
weighted avg       0.92      0.90      0.90       200


Confusion Matrix:
 [[92 20]
 [ 0 88]]


### Extra Task - Implement a SPAM/HAM classifier

https://www.kaggle.com/t/b384e34013d54d238490103bc3c360ce

The classifier can not be changed!!! It must be the MultinomialNB with default parameters!

Your task is to **find the most relevant features**.

For example, you can test the following options and check which of them performs better:
- Using "Bag of Words" only
- Using "TF-IDF" only
- Bag of Words + extra flags (money_mark, suspicious_words, text_len)
- TF-IDF + extra flags


You can work with teams of two persons (recommended).

In [47]:
from sklearn.model_selection import train_test_split

X = data[["text", "money_mark", "suspicious_words", "text_len"]]
y = data["label"]

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [48]:
from sklearn.metrics import classification_report, f1_score, accuracy_score
import numpy as np

def evaluate(name, y_true, y_pred):
    f1_spam = f1_score(y_true, y_pred, pos_label=1)
    acc = accuracy_score(y_true, y_pred)
    print(f"\n{name}")
    print(f"Accuracy: {acc:.4f}")
    print(f"F1 (spam=1): {f1_spam:.4f}")
    print(classification_report(y_true, y_pred, digits=4))
    return f1_spam

In [49]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.preprocessing import MinMaxScaler
from scipy.sparse import hstack, csr_matrix

# Prepare extra numeric features (scaled) - fit scaler on train only
scaler = MinMaxScaler()

train_extra = X_train[["money_mark", "suspicious_words", "text_len"]].values
val_extra   = X_val[["money_mark", "suspicious_words", "text_len"]].values

train_extra_scaled = scaler.fit_transform(train_extra)
val_extra_scaled   = scaler.transform(val_extra)

# Convert to sparse so we can hstack with sparse text matrices
train_extra_sp = csr_matrix(train_extra_scaled)
val_extra_sp   = csr_matrix(val_extra_scaled)

results = {}

# --- A) Bag of Words only ---
bow = CountVectorizer()
Xtr_bow = bow.fit_transform(X_train["text"])
Xva_bow = bow.transform(X_val["text"])

nb = MultinomialNB()  # default params (REQUIRED)
nb.fit(Xtr_bow, y_train)
pred = nb.predict(Xva_bow)
results["BoW only"] = evaluate("BoW only", y_val, pred)

# --- B) TF-IDF only ---
tfidf = TfidfVectorizer()
Xtr_tfidf = tfidf.fit_transform(X_train["text"])
Xva_tfidf = tfidf.transform(X_val["text"])

nb = MultinomialNB()  # default params (REQUIRED)
nb.fit(Xtr_tfidf, y_train)
pred = nb.predict(Xva_tfidf)
results["TF-IDF only"] = evaluate("TF-IDF only", y_val, pred)

# --- C) BoW + extra flags ---
Xtr_bow_plus = hstack([Xtr_bow, train_extra_sp])
Xva_bow_plus = hstack([Xva_bow, val_extra_sp])

nb = MultinomialNB()  # default params (REQUIRED)
nb.fit(Xtr_bow_plus, y_train)
pred = nb.predict(Xva_bow_plus)
results["BoW + extra"] = evaluate("BoW + extra", y_val, pred)

# --- D) TF-IDF + extra flags ---
Xtr_tfidf_plus = hstack([Xtr_tfidf, train_extra_sp])
Xva_tfidf_plus = hstack([Xva_tfidf, val_extra_sp])

nb = MultinomialNB()  # default params (REQUIRED)
nb.fit(Xtr_tfidf_plus, y_train)
pred = nb.predict(Xva_tfidf_plus)
results["TF-IDF + extra"] = evaluate("TF-IDF + extra", y_val, pred)

# Show best configuration
best_name = max(results, key=results.get)
print("\n=======================")
print("Best feature set:", best_name)
print("Best F1 (spam=1):", results[best_name])
print("=======================")


BoW only
Accuracy: 0.9700
F1 (spam=1): 0.9670
              precision    recall  f1-score   support

           0     1.0000    0.9464    0.9725       112
           1     0.9362    1.0000    0.9670        88

    accuracy                         0.9700       200
   macro avg     0.9681    0.9732    0.9698       200
weighted avg     0.9719    0.9700    0.9701       200


TF-IDF only
Accuracy: 0.9450
F1 (spam=1): 0.9412
              precision    recall  f1-score   support

           0     1.0000    0.9018    0.9484       112
           1     0.8889    1.0000    0.9412        88

    accuracy                         0.9450       200
   macro avg     0.9444    0.9509    0.9448       200
weighted avg     0.9511    0.9450    0.9452       200


BoW + extra
Accuracy: 0.9700
F1 (spam=1): 0.9670
              precision    recall  f1-score   support

           0     1.0000    0.9464    0.9725       112
           1     0.9362    1.0000    0.9670        88

    accuracy                       

In [50]:
import numpy as np

# Example: if your best model was TF-IDF only (nb, tfidf exist from above)
# If your best was BoW, use bow instead.

feature_names = tfidf.get_feature_names_out()  # or bow.get_feature_names_out()
log_probs = nb.feature_log_prob_               # shape: (2, n_features_text [+ extras])

# If you added extras, the last 3 columns are extras, so only take text part:
n_text_features = len(feature_names)
spam_logp = log_probs[1, :n_text_features]  # class 1 = spam

top_idx = np.argsort(spam_logp)[-20:]
top_words = feature_names[top_idx]

print("Top words most associated with SPAM (by NB log prob):")
print(top_words[::-1])

Top words most associated with SPAM (by NB log prob):
['money' 'account' 'bank' 'fund' 'transaction' 'mr' 'country' 'business'
 'company' 'transfer' 'kin' 'million' 'next' 'please' 'name' 'contact'
 'foreign' 'father' 'dollar' 'security']
